# Thermal Stack

Pipeline dans `thermal_stack.py`. Coûts marginaux en **JPY/kWh** alignés sur le spot JEPX.

In [ ]:
import pandas as pd
from IPython.display import display

from thermal_stack import (
    ThermalStackConfig,
    MarginalCostConfig,
    StackAnalysisConfig,
    build_thermal_stack,
    merit_order_for_areas,
    plot_merit_order_plotly,
    plot_regional_merit_order_plotly,
    identify_price_setters,
    summarize_price_setters,
    plot_price_setter_share_plotly,
    PriceSetterConfig,
    default_price_setter_configs,
    default_marginal_cost_configs,
    default_parallel_stack_configs,
    run_parallel_stack_analysis,
    summarize_parallel_stack_analysis,
    compare_price_setter_demand_levels,
    plot_price_setter_config_comparison,
    COST_METHOD_OPERATIONAL_JPY,
    COST_METHOD_ARTICLE_JPY,
    LNG_PRICE_JLC,
    KANSAI_FENCE_AREAS,
)
import data_loader

In [ ]:
operational_cost = MarginalCostConfig(
    "operational_jlc_jpy",
    COST_METHOD_OPERATIONAL_JPY,
    LNG_PRICE_JLC,
)
article_cost = MarginalCostConfig(
    "article_jpy",
    COST_METHOD_ARTICLE_JPY,
    LNG_PRICE_JLC,
    zero_mc_fuels=("Hydro",),
)
article_vre_cost = MarginalCostConfig(
    "article_jpy_vre",
    COST_METHOD_ARTICLE_JPY,
    LNG_PRICE_JLC,
    zero_mc_fuels=("Hydro", "Solar", "Wind"),
)

result = build_thermal_stack(
    ThermalStackConfig(
        delivery_month="2026-04",
        export_excel=False,
        load_unit_production=False,
        marginal_cost_config=operational_cost,
    )
)
print(f"Delivery month: {result.delivery_month:%Y-%m}")
print(f"Thermal stack rows: {len(result.merit_order):,}")
display(result.fuel_cocktail[[c for c in result.fuel_cocktail.columns if 'jpy' in c.lower()]].tail(3))

## Merit Order (JPY/kWh)

In [ ]:
region = "Kansai"
mo_region = merit_order_for_areas(result, areas=region, cost_config=operational_cost)
median_snapshot = identify_price_setters(mo_region, config=PriceSetterConfig(
    "Kansai | consumption", ("Kansai",), "Kansai", "consumption"
), start=result.delivery_month, end=result.delivery_month + pd.offsets.MonthEnd(0))

fig = plot_merit_order_plotly(
    mo_region,
    region=region,
    period=result.delivery_month,
    demand_gw=median_snapshot["demand_gw"].median(),
    spot_price_jpy_kwh=median_snapshot["spot_jpy_kwh"].median(),
    title=f"Merit Order ({region}) - JPY/kWh - {result.delivery_month:%B %Y}",
)
fig.show()

## Modes paralleles

Combinaison de:
- **Cout marginal**: operational JLC / article (alpha + C_f) / article + VRE (hydro, solar, wind a MC=0)
- **Region + demande**: Kansai vs fence Kansai+Hokuriku+Chubu x consumption / residual_load / thermal_residual

Prix spot et stack en **JPY/kWh**.

In [ ]:
parallel_cost_modes = [operational_cost, article_cost, article_vre_cost]
parallel_region_modes = default_price_setter_configs(include_kansai_fence=True)
parallel_configs = default_parallel_stack_configs(
    include_kansai_fence=True,
    cost_configs=parallel_cost_modes,
    price_setter_configs=parallel_region_modes,
)

parallel_results = run_parallel_stack_analysis(
    result,
    analysis_configs=parallel_configs,
    start=result.delivery_month,
    end=result.delivery_month + pd.offsets.MonthEnd(0),
)
parallel_summary = summarize_parallel_stack_analysis(parallel_results)
display(parallel_summary.round(3))

In [ ]:
# Comparer operational vs article sur Kansai | thermal_residual
focus = parallel_summary[
    parallel_summary["analysis"].str.contains("thermal_residual")
    & parallel_summary["areas"].str.contains("Kansai")
]
fig_focus = plot_price_setter_config_comparison(
    focus,
    title=f"Kansai thermal residual - cost modes - {result.delivery_month:%B %Y}",
)
fig_focus.show()